In [1]:
import glob
import tifffile
import numpy as np
from pathlib import Path
import dask.array as da

In [3]:
def _process_tif(chunck, output_dir, compression="none", level=0, forground_threshold=0):
    output_dir.mkdir(parents=True, exist_ok=True)
    
    filename = chunck[0]  # 获取文件名
    image = tifffile.imread(filename)
    image[image<forground_threshold] = 0
    output_file_name = output_dir / Path(filename).name

    tifffile.imwrite(
        output_file_name,
        image,
        compression=compression,
        compressionargs=(
            {"level": level} if compression in ("zlib", "adobe_deflate") else None
        ),
        photometric="minisblack",
        metadata={"axes": "YX"},
    )
    return output_file_name

def copy_tiff_using_dask(tiff_dir_name:str, output_dir_name:str, compression="none", level=0, forground_threshold=0):
    tiff_name_format = Path(tiff_dir_name) / "*.tif"
    filenames = da.from_array(glob.glob(str(tiff_name_format)), chunks=(1,))
    output_dir = Path(output_dir_name)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_filenames = filenames.map_blocks(
        _process_tif,
        dtype=object,  # 输出类型为对象（字符串）
        meta=np.array([], dtype=object),  # 元数据定义
        chunks=(1,),  # 输出分块形状为 1D（每个分块生成一个文件名）
        output_dir=output_dir,
        compression=compression,
        level=level,
        forground_threshold=forground_threshold,
    ).compute()
    # filenames.map_blocks(_process_tif, output_dir=output_dir, compression=compression, level=level, forground_threshold=forground_threshold).compute()


In [19]:
tiff_dir_name = r"\\192.168.1.192/worm-tools/Jinghao-Wang/tea_experiment/20250421_EGCG/w1_2025-04-21_11-39-40/"

In [10]:
copy_tiff_using_dask(
    tiff_dir_name = tiff_dir_name,
    output_dir_name=r"H:\Process_temporary\WJH\olfactory\ID\image_data\20250421_EGCG\w1",
    compression="zlib",
    level=1,
    forground_threshold=120,
)

In [11]:
import os
import numpy as np
from tqdm import tqdm
import dask.array as da
import napari
import tifffile

def _read_frame(filenames_vol,**kwargs):
    return tifffile.imread( filenames_vol[0][0] )[np.newaxis,np.newaxis,:,:]

def extract_volume_numbers_from_dir(image_dir, frames_per_volume=20):
    volume_numbers = []
    frame_numbers = (
        {}
    )  # key: volume number, value: a set of frame numbers in the volume

    # Get a list of all files in the directory
    files = os.listdir(image_dir)

    # Sort the files in ascending order
    files.sort()

    # Iterate over the files
    for file in files:
        # Check if the file is a tiff image
        if file.endswith(".tif"):
            # Extract the frame number from the file name
            frame_number = int(file.split(".")[0])
            cur_volume_number = frame_number // frames_per_volume
            # add the frame number to the set of frame numbers for the current volume
            if cur_volume_number not in frame_numbers:
                frame_numbers[cur_volume_number] = set()
            frame_numbers[cur_volume_number].add(frame_number)

    # Iterate over the frame numbers
    for volume_number, frame_number_set in frame_numbers.items():
        # Check if the frame numbers are continuous
        if len(frame_number_set) == frames_per_volume:
            volume_numbers.append(volume_number)

    return volume_numbers


def get_filenames_vols(
    volume_numbers,
    tiff_root_path,
    img_height,
    img_width,
    img_dtype,
    frame_number_per_volume,
    z_start_frame_number,
    z_end_frame_number,
    mod2_reverse,
    show_progress=False,  # New parameter to control tqdm progress bar
):
    img_depth = z_end_frame_number - z_start_frame_number + 1
    # volumes_img = np.zeros(
    #     (len(volume_numbers), img_depth, img_height, img_width), dtype=img_dtype
    # )
    filenames_vols = []
    iterable = tqdm(volume_numbers) if show_progress else volume_numbers
    for index, volume_number in enumerate(iterable):
        filenames_frames = []
        frame_numbers = list(
            range(
                volume_number * frame_number_per_volume + z_start_frame_number,
                volume_number * frame_number_per_volume + z_end_frame_number + 1,
            )
        )
        if mod2_reverse[volume_number % 2]:
            frame_numbers = frame_numbers[::-1]
        for frame_number in frame_numbers:
            tiff_file_name = f"{frame_number:08d}.tif"
            # read tiff file and place them in volume_img
            tiff_file_path = os.path.join(tiff_root_path, tiff_file_name)
            filenames_frames.append(tiff_file_path)
            # volumes_img[index, frame_numbers.index(frame_number), :, :] = plt.imread(
            #     tiff_file_path
            # )
        filenames_vols.append(filenames_frames)
    return filenames_vols

In [12]:
def lazy_read_tiff_stack(
    tiff_path_,
    volume_read_params,
):
    # Extract volume numbers from the directory
    vols = extract_volume_numbers_from_dir(tiff_path_)
    file_names = get_filenames_vols(
        volume_numbers=vols,
        tiff_root_path=tiff_path_,
        show_progress=True,
        **volume_read_params,
    )
    valid_frames_per_volume = volume_read_params["z_end_frame_number"] - volume_read_params["z_start_frame_number"] + 1
    file_names_dask = da.from_array(file_names, chunks=(1, 1))
    images_dask = file_names_dask.map_blocks(
        _read_frame,
        chunks=da.core.normalize_chunks((1, 1, 1024, 1024), (len(vols), valid_frames_per_volume, 1024, 1024)),
        # multiple_files=True,
        new_axis=[2, 3],
        meta=np.array((), dtype=np.uint16),  # meta overwrites `dtype` argument
    )
    return images_dask


## 0421_EGCG

In [ ]:
# w1

In [32]:
tiff_dir_name = r"\\192.168.1.192/worm-tools/Jinghao-Wang/tea_experiment/20250421_EGCG/w1_2025-04-21_11-39-40/"

In [20]:
exp_path = tiff_dir_name
red_tiff_path_ = rf"{exp_path}\0_Camera-Red_VSC-10629"
green_tiff_path_ = rf"{exp_path}\1_Camera-Green_VSC-09321"
volume_read_params = dict(
    z_start_frame_number=0,
    z_end_frame_number=17,
    mod2_reverse=[False, False],
    img_width=1024,
    img_height=1024,
    frame_number_per_volume=20,
    img_dtype=np.uint16,
)
red = lazy_read_tiff_stack(red_tiff_path_, volume_read_params)
green = lazy_read_tiff_stack(green_tiff_path_, volume_read_params)

100%|██████████| 3283/3283 [00:00<00:00, 15918.67it/s]


In [34]:
def save_dask_array_as_npy(dask_array, output_path):
    # Convert the Dask array to a NumPy array
    numpy_array = dask_array.compute()
    # Ensure the output directory exists
    output_dir = Path(output_path).parent
    output_dir.mkdir(parents=True, exist_ok=True)
    # Save the NumPy array to a .npy file
    np.save(output_path, numpy_array)

In [30]:
red_path = r"H:\Process_temporary\WJH\olfactory\ID\image_data\20250421_EGCG\w1\red.npy"

In [31]:
save_dask_array_as_npy(red, red_path)

In [27]:
import napari
viewer = napari.Viewer()
viewer.add_image(red, name="red")

<Image layer 'red' at 0x2dd76b9e6d0>

In [24]:
green = green[:20]

In [33]:
save_dask_array_as_npy(green, r"H:\Process_temporary\WJH\olfactory\ID\image_data\20250421_EGCG\w1\green.npy")

In [ ]:
# w2

In [35]:
tiff_dir_name = r"//192.168.1.192/worm-tools/Jinghao-Wang/tea_experiment/20250421_EGCG/w2_2025-04-21_12-11-33"

In [36]:
exp_path = tiff_dir_name
red_tiff_path_ = rf"{exp_path}\0_Camera-Red_VSC-10629"
green_tiff_path_ = rf"{exp_path}\1_Camera-Green_VSC-09321"
volume_read_params = dict(
    z_start_frame_number=0,
    z_end_frame_number=17,
    mod2_reverse=[False, False],
    img_width=1024,
    img_height=1024,
    frame_number_per_volume=20,
    img_dtype=np.uint16,
)
red = lazy_read_tiff_stack(red_tiff_path_, volume_read_params)
green = lazy_read_tiff_stack(green_tiff_path_, volume_read_params)

100%|██████████| 3303/3303 [00:00<00:00, 14713.29it/s]


In [37]:
save_dask_array_as_npy(red, r"H:\Process_temporary\WJH\olfactory\ID\image_data\20250421_EGCG\w2\red.npy")
save_dask_array_as_npy(green[:10], r"H:\Process_temporary\WJH\olfactory\ID\image_data\20250421_EGCG\w2\green.npy")

### w3

In [40]:
tiff_dir_name = r"//192.168.1.192/worm-tools/Jinghao-Wang/tea_experiment/20250421_EGCG/w3_2025-04-21_13-51-08"

In [41]:
exp_path = tiff_dir_name
red_tiff_path_ = rf"{exp_path}\0_Camera-Red_VSC-10629"
green_tiff_path_ = rf"{exp_path}\1_Camera-Green_VSC-09321"
volume_read_params = dict(
    z_start_frame_number=0,
    z_end_frame_number=17,
    mod2_reverse=[False, False],
    img_width=1024,
    img_height=1024,
    frame_number_per_volume=20,
    img_dtype=np.uint16,
)
red = lazy_read_tiff_stack(red_tiff_path_, volume_read_params)
green = lazy_read_tiff_stack(green_tiff_path_, volume_read_params)

100%|██████████| 3289/3289 [00:00<00:00, 15482.49it/s]


In [42]:
save_dask_array_as_npy(red, r"H:\Process_temporary\WJH\olfactory\ID\image_data\20250421_EGCG\w3\red.npy")
save_dask_array_as_npy(green[:10], r"H:\Process_temporary\WJH\olfactory\ID\image_data\20250421_EGCG\w3\green.npy")

In [43]:
# w4
tiff_dir_name = r"//192.168.1.192/worm-tools/Jinghao-Wang/tea_experiment/20250421_EGCG/w4_2025-04-21_14-23-21"

In [44]:
exp_path = tiff_dir_name
red_tiff_path_ = rf"{exp_path}\0_Camera-Red_VSC-10629"
green_tiff_path_ = rf"{exp_path}\1_Camera-Green_VSC-09321"
volume_read_params = dict(
    z_start_frame_number=0,
    z_end_frame_number=17,
    mod2_reverse=[False, False],
    img_width=1024,
    img_height=1024,
    frame_number_per_volume=20,
    img_dtype=np.uint16,
)
red = lazy_read_tiff_stack(red_tiff_path_, volume_read_params)
green = lazy_read_tiff_stack(green_tiff_path_, volume_read_params)

100%|██████████| 3291/3291 [00:00<00:00, 15941.07it/s]


In [45]:
save_dask_array_as_npy(red, r"H:\Process_temporary\WJH\olfactory\ID\image_data\20250421_EGCG\w4\red.npy")
save_dask_array_as_npy(green[:10], r"H:\Process_temporary\WJH\olfactory\ID\image_data\20250421_EGCG\w4\green.npy")

In [46]:
# w5
tiff_dir_name = r"//192.168.1.192/worm-tools/Jinghao-Wang/tea_experiment/20250421_EGCG/w5_2025-04-21_14-56-54"

In [47]:
exp_path = tiff_dir_name
red_tiff_path_ = rf"{exp_path}\0_Camera-Red_VSC-10629"
green_tiff_path_ = rf"{exp_path}\1_Camera-Green_VSC-09321"
volume_read_params = dict(
    z_start_frame_number=0,
    z_end_frame_number=17,
    mod2_reverse=[False, False],
    img_width=1024,
    img_height=1024,
    frame_number_per_volume=20,
    img_dtype=np.uint16,
)
red = lazy_read_tiff_stack(red_tiff_path_, volume_read_params)
green = lazy_read_tiff_stack(green_tiff_path_, volume_read_params)

100%|██████████| 3267/3267 [00:00<00:00, 15783.37it/s]


In [48]:
save_dask_array_as_npy(red, r"H:\Process_temporary\WJH\olfactory\ID\image_data\20250421_EGCG\w5\red.npy")
save_dask_array_as_npy(green[:10], r"H:\Process_temporary\WJH\olfactory\ID\image_data\20250421_EGCG\w5\green.npy")

In [49]:
# w6
tiff_dir_name = r"//192.168.1.192/worm-tools/Jinghao-Wang/tea_experiment/20250421_EGCG/w6_2025-04-21_16-13-49"

In [50]:
exp_path = tiff_dir_name
red_tiff_path_ = rf"{exp_path}\0_Camera-Red_VSC-10629"
green_tiff_path_ = rf"{exp_path}\1_Camera-Green_VSC-09321"
volume_read_params = dict(
    z_start_frame_number=0,
    z_end_frame_number=17,
    mod2_reverse=[False, False],
    img_width=1024,
    img_height=1024,
    frame_number_per_volume=20,
    img_dtype=np.uint16,
)
red = lazy_read_tiff_stack(red_tiff_path_, volume_read_params)
green = lazy_read_tiff_stack(green_tiff_path_, volume_read_params)

100%|██████████| 3294/3294 [00:00<00:00, 15334.69it/s]


In [51]:
save_dask_array_as_npy(red, r"H:\Process_temporary\WJH\olfactory\ID\image_data\20250421_EGCG\w6\red.npy")
save_dask_array_as_npy(green[:10], r"H:\Process_temporary\WJH\olfactory\ID\image_data\20250421_EGCG\w6\green.npy")

## batch process whole folder

In [55]:
def batch_process_folder(folder_path, output_path):
    # Get all subfolders in the directory
    subfolders = [f for f in Path(folder_path).iterdir() if f.is_dir()]
    
    for subfolder in subfolders:
        # Process each subfolder
        # if subfolder startswith("w"):
        if subfolder.name.startswith("w"):
            # Process the subfolder
            print(f"Processing {subfolder}...")
            tiff_dir_name = os.path.join(folder_path, subfolder.name)
            exp_path = tiff_dir_name
            red_tiff_path_ = rf"{exp_path}\0_Camera-Red_VSC-10629"
            green_tiff_path_ = rf"{exp_path}\1_Camera-Green_VSC-09321"
            volume_read_params = dict(
                z_start_frame_number=0,
                z_end_frame_number=17,
                mod2_reverse=[False, False],
                img_width=1024,
                img_height=1024,
                frame_number_per_volume=20,
                img_dtype=np.uint16,
            )
            red = lazy_read_tiff_stack(red_tiff_path_, volume_read_params)
            green = lazy_read_tiff_stack(green_tiff_path_, volume_read_params)
            save_dask_array_as_npy(red[:10], os.path.join(output_path, subfolder.name, "red.npy"))
            save_dask_array_as_npy(green[:10], os.path.join(output_path, subfolder.name, "green.npy"))


In [56]:
# 20250421_EGCG_high
nas_folder_path = rf"//192.168.1.192/worm-tools/Jinghao-Wang/tea_experiment/20250421_EGCG_high"
save_folder_path = rf"H:\Process_temporary\WJH\olfactory\ID\image_data\20250421_EGCG_high"
batch_process_folder(nas_folder_path, save_folder_path)

Processing \\192.168.1.192\worm-tools\Jinghao-Wang\tea_experiment\20250421_EGCG_high\w1_2025-04-21_17-21-34...


100%|██████████| 2027/2027 [00:00<00:00, 15177.22it/s]


Processing \\192.168.1.192\worm-tools\Jinghao-Wang\tea_experiment\20250421_EGCG_high\w2_2025-04-21_18-00-51...


100%|██████████| 2018/2018 [00:00<00:00, 15167.67it/s]


In [57]:
# 20250422_quinine
nas_folder_path = r"//192.168.1.192/worm-tools/Jinghao-Wang/tea_experiment/20250422_quinine"
save_folder_path = r"H:\Process_temporary\WJH\olfactory\ID\image_data\20250422_quinine"
batch_process_folder(nas_folder_path, save_folder_path)

Processing \\192.168.1.192\worm-tools\Jinghao-Wang\tea_experiment\20250422_quinine\w1_2025-04-22_16-42-03...


100%|██████████| 2253/2253 [00:00<00:00, 14992.85it/s]


Processing \\192.168.1.192\worm-tools\Jinghao-Wang\tea_experiment\20250422_quinine\w2_2025-04-22_17-05-57...


100%|██████████| 3348/3348 [00:00<00:00, 15574.03it/s]


Processing \\192.168.1.192\worm-tools\Jinghao-Wang\tea_experiment\20250422_quinine\w3_2025-04-22_18-10-14...


100%|██████████| 3285/3285 [00:00<00:00, 14943.51it/s]


Processing \\192.168.1.192\worm-tools\Jinghao-Wang\tea_experiment\20250422_quinine\w4_2025-04-22_18-36-02...


100%|██████████| 3382/3382 [00:00<00:00, 14959.66it/s]


In [58]:
# 20250422_quinine_high
nas_folder_path = r'//192.168.1.192/worm-tools/Jinghao-Wang/tea_experiment/20250422_quinine_high'
save_folder_path = r'H:\Process_temporary\WJH\olfactory\ID\image_data\20250422_quinine_high'
batch_process_folder(nas_folder_path, save_folder_path)

Processing \\192.168.1.192\worm-tools\Jinghao-Wang\tea_experiment\20250422_quinine_high\w1_2025-04-22_19-22-52...


100%|██████████| 3057/3057 [00:00<00:00, 15601.18it/s]


Processing \\192.168.1.192\worm-tools\Jinghao-Wang\tea_experiment\20250422_quinine_high\w2_2025-04-22_19-39-39...


100%|██████████| 3261/3261 [00:00<00:00, 15077.56it/s]


Processing \\192.168.1.192\worm-tools\Jinghao-Wang\tea_experiment\20250422_quinine_high\w3_2025-04-23_09-08-36...


100%|██████████| 3262/3262 [00:00<00:00, 14686.76it/s]


Processing \\192.168.1.192\worm-tools\Jinghao-Wang\tea_experiment\20250422_quinine_high\w4_2025-04-23_09-30-14...


100%|██████████| 3282/3282 [00:00<00:00, 16263.98it/s]


In [60]:
# 20250428_TF
nas_folder_path = r"//192.168.1.192/worm-tools/Jinghao-Wang/tea_experiment/20250428_TF"
save_folder_path = r"H:\Process_temporary\WJH\olfactory\ID\image_data\20250428_TF"
batch_process_folder(nas_folder_path, save_folder_path)

Processing \\192.168.1.192\worm-tools\Jinghao-Wang\tea_experiment\20250428_TF\w1_2025-04-28_14-22-35...


100%|██████████| 3285/3285 [00:00<00:00, 16137.49it/s]


Processing \\192.168.1.192\worm-tools\Jinghao-Wang\tea_experiment\20250428_TF\w2_2025-04-28_14-44-02...


100%|██████████| 3249/3249 [00:00<00:00, 16133.43it/s]


Processing \\192.168.1.192\worm-tools\Jinghao-Wang\tea_experiment\20250428_TF\w3_2025-04-28_15-14-33...


...

100%|██████████| 3251/3251 [00:00<00:00, 16386.36it/s]


Processing \\192.168.1.192\worm-tools\Jinghao-Wang\tea_experiment\20250428_TF\w6_2025-04-28_17-09-26...


100%|██████████| 3344/3344 [00:00<00:00, 16066.76it/s]


Processing \\192.168.1.192\worm-tools\Jinghao-Wang\tea_experiment\20250428_TF\w7_2025-04-28_17-36-05...


100%|██████████| 3259/3259 [00:00<00:00, 15867.65it/s]


In [62]:
# 20250428_compare
nas_folder_path = r"//192.168.1.192/worm-tools/Jinghao-Wang/tea_experiment/20250428_comparison"
save_folder_path = r"H:\Process_temporary\WJH\olfactory\ID\image_data\20250428_compararison"
batch_process_folder(nas_folder_path, save_folder_path)

Processing \\192.168.1.192\worm-tools\Jinghao-Wang\tea_experiment\20250428_comparison\w1_2025-04-28_20-26-48...


100%|██████████| 2399/2399 [00:00<00:00, 15301.18it/s]


Processing \\192.168.1.192\worm-tools\Jinghao-Wang\tea_experiment\20250428_comparison\w2_2025-04-28_20-50-33...


100%|██████████| 2693/2693 [00:00<00:00, 15614.95it/s]


Processing \\192.168.1.192\worm-tools\Jinghao-Wang\tea_experiment\20250428_comparison\w3_2025-04-28_21-16-17...


...

100%|██████████| 2274/2274 [00:00<00:00, 15563.92it/s]


Processing \\192.168.1.192\worm-tools\Jinghao-Wang\tea_experiment\20250428_comparison\w5_2025-04-28_22-00-26...


100%|██████████| 2274/2274 [00:00<00:00, 15203.34it/s]


Processing \\192.168.1.192\worm-tools\Jinghao-Wang\tea_experiment\20250428_comparison\w6_2025-04-28_22-12-25...


100%|██████████| 2258/2258 [00:00<00:00, 14613.38it/s]


In [63]:
# 20250429_caffeine
nas_folder_path = r"//192.168.1.192/worm-tools/Jinghao-Wang/tea_experiment/20250429_caffeine"
save_folder_path = r"H:\Process_temporary\WJH\olfactory\ID\image_data\20250429_caffeine"
batch_process_folder(nas_folder_path, save_folder_path)

Processing \\192.168.1.192\worm-tools\Jinghao-Wang\tea_experiment\20250429_caffeine\w1_2025-04-29_14-52-59...


100%|██████████| 3268/3268 [00:00<00:00, 15537.21it/s]


Processing \\192.168.1.192\worm-tools\Jinghao-Wang\tea_experiment\20250429_caffeine\w2_2025-04-29_15-13-01...


100%|██████████| 3256/3256 [00:00<00:00, 16195.57it/s]


Processing \\192.168.1.192\worm-tools\Jinghao-Wang\tea_experiment\20250429_caffeine\w3_2025-04-29_16-11-21...


...

100%|██████████| 3260/3260 [00:00<00:00, 15353.06it/s]


Processing \\192.168.1.192\worm-tools\Jinghao-Wang\tea_experiment\20250429_caffeine\w6_2025-04-29_23-43-57...


100%|██████████| 3231/3231 [00:00<00:00, 15092.31it/s]


Processing \\192.168.1.192\worm-tools\Jinghao-Wang\tea_experiment\20250429_caffeine\w7_2025-04-30_00-03-00...


100%|██████████| 3245/3245 [00:00<00:00, 15949.12it/s]
